# Description

In this notebook, I will explore the benchmark Human Eval and using DeepSeek to generate it.

In [18]:
import os 
import sys
import numpy as np 
import pandas as pd 
import re
import io
import re
import ast
import types
import time
import unittest
import importlib
from typing import List, Tuple, Dict, Any, Set
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import transformers

In [2]:
model_name = "deepseek-ai/deepseek-coder-6.7b-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name,\
        trust_remote_code=True).cuda()

MAX_NEW_TOKENS = 512

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

# 1. Load data

In [3]:
PATH_CSV_DATA = "data/raw_data/human_eval.csv"

In [4]:
df = pd.read_csv(PATH_CSV_DATA)
print(f"Dataframe shape: {df.shape}")
df.sample(1)

Dataframe shape: (164, 5)


,task_id,prompt,canonical_solution,test,entry_point
119,HumanEval/119,\ndef match_parens(lst):\n '''\n You are...,def check(s):\n val = 0\n fo...,def check(candidate):\n\n # Check some simp...,match_parens


In [5]:
idx = np.random.randint(0, df.shape[0])

code_description = df.loc[idx, "prompt"]
test_case = df.loc[idx, "test"]
entry_point = df.loc[idx, "entry_point"]

print("Code description:")
print(code_description)
print("=" * 20)
print("Test Case:")
print(test_case)

Code description:

def multiply(a, b):
    """Complete the function that takes two integers and returns 
    the product of their unit digits.
    Assume the input is always valid.
    Examples:
    multiply(148, 412) should return 16.
    multiply(19, 28) should return 72.
    multiply(2020, 1851) should return 0.
    multiply(14,-15) should return 20.
    """

Test Case:
def check(candidate):

    # Check some simple cases
    assert candidate(148, 412) == 16, "First test error: " + str(candidate(148, 412))                    
    assert candidate(19, 28) == 72, "Second test error: " + str(candidate(19, 28))           
    assert candidate(2020, 1851) == 0, "Third test error: " + str(candidate(2020, 1851))
    assert candidate(14,-15) == 20, "Fourth test error: " + str(candidate(14,-15))      
    assert candidate(76, 67) == 42, "Fifth test error: " + str(candidate(76, 67))      
    assert candidate(17, 27) == 49, "Sixth test error: " + str(candidate(17, 27))      


    # Check som

# 2. Using Deepseek to generate sample

## 2.1. Generate code

In [6]:
def extract_function(llm_text):
    code = "Empty code"
    # 1) Grab text between <code>...</code>
    m = re.search(r"<code>\s*(.*?)\s*</code>", llm_text, flags=re.S|re.M)
    if m:
        code = m.group(1)
    else:
    # 2) Optionally, if the model sometimes adds backticks, strip them
        code = re.sub(r"^```(?:python)?\s*|\s*```$", "", code.strip())

    return code

In [7]:
constraints = """
Output only a complete and valid Python code for this function. 
Do not change the provided function signature.
Do NOT print any markdown or include the test cases in your output.
Wrap your output strictly between the markers:
<code>
... your code ...
</code>
"""

input_prompt = f"""write a complete python function
based on the following description:\n{code_description}.\n
with the following constraints:\n{constraints}
"""

print("Input prompt to:")
print(input_prompt)

Input prompt to:
write a complete python function
based on the following description:

def multiply(a, b):
    """Complete the function that takes two integers and returns 
    the product of their unit digits.
    Assume the input is always valid.
    Examples:
    multiply(148, 412) should return 16.
    multiply(19, 28) should return 72.
    multiply(2020, 1851) should return 0.
    multiply(14,-15) should return 20.
    """
.

with the following constraints:

Output only a complete and valid Python code for this function. 
Do not change the provided function signature.
Do NOT print any markdown or include the test cases in your output.
Wrap your output strictly between the markers:
<code>
... your code ...
</code>




In [11]:
def generate_response(input_text, max_new_tokens=MAX_NEW_TOKENS):
    messages=[
        { 'role': 'system', 'content': "You are DeepSeek - a helpful assistant."},
        { 'role': 'user', 'content': f"{input_text}"}
    ]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
    outputs = model.generate(inputs,
                             max_new_tokens=max_new_tokens,
                             num_return_sequences=1,
                             pad_token_id=tokenizer.eos_token_id,
                             eos_token_id=tokenizer.eos_token_id)
    response = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)
    return response

In [13]:
output = generate_response(input_prompt)
print("Response:\n", output)

Response:
 <code>
def multiply(a, b):
    # Extract the unit digits of the numbers
    unit_digit_a = abs(a) % 10
    unit_digit_b = abs(b) % 10

    # Return the product of the unit digits
    return unit_digit_a * unit_digit_b
</code>



We can extract the complete code

In [14]:
completed_code = extract_function(output)
print(f"The complete code:\n")
print(completed_code)

The complete code:

def multiply(a, b):
    # Extract the unit digits of the numbers
    unit_digit_a = abs(a) % 10
    unit_digit_b = abs(b) % 10

    # Return the product of the unit digits
    return unit_digit_a * unit_digit_b


## 2.2. Evaluate the generated code

In [15]:
import builtins
import typing

def create_namespace():
    ns = {}

    # 1. Standard builtins (print, len, etc.)
    ns.update({k: getattr(builtins, k) for k in dir(builtins)})

    # 2. Install common typing names (List, Optional, etc.)
    for name in typing.__all__:
        ns[name] = getattr(typing, name)

    # 3. (Optional) Add math, random, itertools, etc.
    import math, random, itertools, statistics
    ns.update({
        'math': math,
        'random': random,
        'itertools': itertools,
        'statistics': statistics,
    })

    return ns

In [16]:
def evaluate_asserts(generated_code: str, test_code: str, entry_point: str):
    # ns = {}
    ns = create_namespace()
    
    # 1. Exec both code strings
    exec(generated_code, ns)
    exec(test_code, ns)

    candidate = ns[entry_point]     # the model's function
    check_fn = ns["check"]          # original check() function
    
    # 2. Parse the test code AST
    tree = ast.parse(test_code)

    # 3. Find the check() function body
    check_body = None
    for node in tree.body:
        if isinstance(node, ast.FunctionDef) and node.name == "check":
            check_body = node.body
            break

    if check_body is None:
        raise ValueError("check() function not found.")
    
    # 4. Evaluate each assert individually
    results = []
    for idx, stmt in enumerate(check_body):
        if isinstance(stmt, ast.Assert):
            # Convert AST back to executable code
            code = compile(ast.Module([stmt], type_ignores=[]), "<assert>", "exec")
            try:
                exec(code, {**ns, "candidate": candidate})
                results.append(("pass", None))
            except Exception as e:
                results.append(("fail", repr(e)))

    # 5. Compute pass percentage
    total = len(results)
    passed = sum(1 for r, _ in results if r == "pass")
    percentage = passed / total if total > 0 else 0.0

    return {
        "total_asserts": total,
        "passed": passed,
        "percentage": percentage,
        "detail": results
    }

In [17]:
result = evaluate_asserts(completed_code, test_case, entry_point)
print(result)

{'total_asserts': 8, 'passed': 8, 'percentage': 1.0, 'detail': [('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None)]}


# 3. Run through all sample

In [20]:
list_df = []
list_num_out_token = []

start_time = time.time()
for idx in range(df.shape[0]):
    if idx % 10 == 0:
        print(f"Processing idx={idx} / {df.shape[0]}")
    
    try:
        # 1. Prepare input prompt
        code_description = df.loc[idx, "prompt"]
        test_case = df.loc[idx, "test"]
        entry_point = df.loc[idx, "entry_point"]

        input_prompt = f"""write a complete python function
        based on the following description:\n{code_description}.\n
        with the following constraints:\n{constraints}
        """

        output = generate_response(input_prompt)
        completed_code = extract_function(output)
        list_num_out_token.append(len(tokenizer(output).input_ids))
        
        # 3. Evaluate the generated code
        result = evaluate_asserts(completed_code, test_case, entry_point)
        total_asserts = result["total_asserts"]
        passed_asserts = result["passed"]
        percentage = result["percentage"]
        
        list_df.append({
            "description": code_description,
            "generated_code": completed_code,
            "test_case": test_case,
            "entry_point": entry_point,
            "total_asserts": total_asserts,
            "passed_asserts": passed_asserts,
            "percentage": percentage,
        })
    except Exception as e:
        print(f"Error at idx={idx}: {e}")
        continue
    
end_time = time.time()
avg_time_per_example = (end_time - start_time) / len(list_df)
print(f"Average time per example: {avg_time_per_example:.2f} seconds")

avg_output_tokens = sum(list_num_out_token) / len(list_num_out_token)
print(f"Average number of output tokens: {avg_output_tokens:.2f}")

Processing idx=0 / 164


Processing idx=10 / 164
Processing idx=20 / 164
Processing idx=30 / 164
Processing idx=40 / 164
Processing idx=50 / 164
Processing idx=60 / 164
Processing idx=70 / 164
Processing idx=80 / 164
Processing idx=90 / 164
Error at idx=94: invalid syntax (<string>, line 1)
Processing idx=100 / 164
Processing idx=110 / 164
Processing idx=120 / 164
Processing idx=130 / 164
Processing idx=140 / 164
Processing idx=150 / 164
Processing idx=160 / 164
Average time per example: 5.18 seconds
Average number of output tokens: 92.37


In [ ]:
# from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeoutError

# list_df = []

# def process_idx(idx):
#     # 1. Prepare input prompt
#     code_description = df.loc[idx, "prompt"]
#     test_case = df.loc[idx, "test"]
#     entry_point = df.loc[idx, "entry_point"]

#     input_prompt = f"""write a complete python function
#     based on the following description:\n{code_description}.\n
#     with the following constraints:\n{constraints}
#     """

#     output = generate_response(input_prompt)
#     completed_code = extract_function(output)

#     # 3. Evaluate the generated code
#     result = evaluate_asserts(completed_code, test_case, entry_point)
#     total_asserts = result["total_asserts"]
#     passed_asserts = result["passed"]
#     percentage = result["percentage"]

#     return {
#         "description": code_description,
#         "generated_code": completed_code,
#         "test_case": test_case,
#         "entry_point": entry_point,
#         "total_asserts": total_asserts,
#         "passed_asserts": passed_asserts,
#         "percentage": percentage,
#     }

# with ThreadPoolExecutor(max_workers=1) as executor:
#     for idx in range(df.shape[0]):
#         if idx % 10 == 0:
#             print(f"Processing idx={idx}/{df.shape[0]}")

#         future = executor.submit(process_idx, idx)

#         try:
#             row_result = future.result(timeout=30)  # 10-second per-index timeout
#             list_df.append(row_result)
#         except FuturesTimeoutError:
#             print(f"Timeout at idx={idx}, skipping...")
#             continue
#         except Exception as e:
#             print(f"Error at idx={idx}: {e}")
#             continue

In [21]:
output_df = pd.DataFrame(list_df)
print(f'Output dataframe shape: {output_df.shape}')
output_df.sample()

Output dataframe shape: (163, 7)


,description,generated_code,test_case,entry_point,total_asserts,passed_asserts,percentage
161,"\ndef string_to_md5(text):\n """"""\n Given...",import hashlib\n\ndef string_to_md5(text):\n ...,def check(candidate):\n\n # Check some simp...,string_to_md5,5,5,1.0


In [22]:
# Save to CSV
output_df.to_csv("data/generated/human_eval_generated_deepseek.csv", index=False)

## 3.1. Check generated code

In [23]:
average_percentage = output_df["percentage"].mean()
print(f"Average pass percentage over all samples: {average_percentage:.2%}")    

Average pass percentage over all samples: 88.17%
